In [1]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

np.random.seed(42)
n = 1000
y_true = np.clip(np.random.beta(a=8, b=1.5, size=n), 0, 1)
X_dummy = np.zeros((n, 1))

dummy_mean = DummyRegressor(strategy="mean")
dummy_mean.fit(X_dummy, y_true)
pred_mean = dummy_mean.predict(X_dummy)

mae_mean = mean_absolute_error(y_true, pred_mean)
rmse_mean = np.sqrt(mean_squared_error(y_true, pred_mean))
r2_mean = r2_score(y_true, pred_mean)

print("MEAN baseline -> MAE:", mae_mean, "RMSE:", rmse_mean, "R2:", r2_mean)

MEAN baseline -> MAE: 0.08437403815829671 RMSE: 0.10676755125440535 R2: 0.0


In [2]:
dummy_median = DummyRegressor(strategy="median")
dummy_median.fit(X_dummy,y_true)
pred_median = dummy_median.predict(X_dummy)

mae_median = mean_absolute_error(y_true, pred_median)
rmse_median = np.sqrt(mean_squared_error(y_true, pred_median))
r2_median = r2_score(y_true, pred_median)

print("MEDIAN baseline -> MAE:", mae_median, "RMSE:", rmse_median, "R2:", r2_median)

MEDIAN baseline -> MAE: 0.08308413573424395 RMSE: 0.10821832821402474 R2: -0.02736100347764281


In [3]:
import duckdb
import pandas as pd

duckdb.sql("CREATE VIEW duolingo_flagship AS SELECT * FROM read_csv_auto('../../data/duolingo_flagship_v4.csv')")
df = duckdb.sql("SELECT * FROM duolingo_flagship").df()

# Load the split saved in part 11 instead of re-deriving it here.
split = pd.read_csv("../../data/split_users.csv")
cv_users = set(split.loc[split["split"] == "cv", "user_id"])
df_cv = df[df["user_id"].isin(cv_users)]

y_cv = df_cv["p_recall"].values
X_dummy_cv = np.zeros((len(y_cv), 1))

print("CV pool rows:", len(y_cv), "| CV users:", df_cv["user_id"].nunique())
print("mean:", y_cv.mean(), "| median:", np.median(y_cv))

CV pool rows: 14438 | CV users: 2125
mean: 0.894244498677102 | median: 1.0


In [4]:
dummy_mean = DummyRegressor(strategy="mean")
dummy_mean.fit(X_dummy_cv,y_cv)
dummy_median = DummyRegressor(strategy="median")
dummy_median.fit(X_dummy_cv,y_cv)

pred_mean = dummy_mean.predict(X_dummy_cv)
pred_median = dummy_median.predict(X_dummy_cv)

mae_mean = mean_absolute_error(y_cv, pred_mean)
rmse_mean = np.sqrt(mean_squared_error(y_cv ,pred_mean))
r2_mean = r2_score(y_cv, pred_mean)

mae_median = mean_absolute_error(y_cv, pred_median)
rmse_median = np.sqrt(mean_squared_error(y_cv, pred_median))
r2_median = r2_score(y_cv, pred_median)

print("MEAN baseline -> MAE:", mae_mean, "RMSE:", rmse_mean, "R2:", r2_mean)
print("MEDIAN baseline -> MAE:", mae_median, "RMSE:", rmse_median, "R2:", r2_median)

MEAN baseline -> MAE: 0.17710360561966873 RMSE: 0.27554148130022776 R2: 0.0
MEDIAN baseline -> MAE: 0.10575550132289792 RMSE: 0.2951395161227673 R2: -0.14730990823328716


## Baseline and metric

Baselines on the CV pool (2,125 users, n=14,438), using the split saved in part 11:
- Mean strategy: MAE 0.177, RMSE 0.276, R² 0.0 (that's not a coincidence ,R²'s zero point is literally the mean baseline)
- Median strategy: MAE 0.106, RMSE 0.295, R² -0.147

Median wins on MAE because p_recall's median is exactly 1.0, so it matches over half the rows exactly. Mean wins on RMSE since mean is the point that minimizes squared error by construction, median isn't built for that.

Going with RMSE as the main metric. It squares errors, so big misses cost a lot more than small ones. That fits how this would actually get used predicting p_recall way too high (model says "will remember" when the user's actually going to forget) is a real learning failure, not just a rounding error. MAE gets reported too since it's easier to read and tells a slightly different story.

Baseline to beat: RMSE 0.276 (the mean strategy, which is also the harder baseline here).

One thing to flag for later: MAE and RMSE both punish over and under-prediction the same way, but the real cost probably isn't symmetric. Overestimating recall is worse than underestimating it. Something to look at with an asymmetric loss down the line.

Note: these numbers replace an earlier run (mean RMSE 0.279 on n=14,125). That run used a split derived inline from row order, which part 14's SQL join later invalidated. Same method, same seed, now measured on the split that's actually saved to disk.